# Assistente contextualizado com LangChain

O pipeline combina pergunta anonimizada, consulta estruturada e protocolos recuperados. O exemplo usa o backend determinístico para ser executável sem GPU; a interface é a mesma utilizada pelo adaptador LoRA.

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
subprocess.run([sys.executable, str(ROOT / "scripts/init_database.py")], cwd=ROOT, check=True)

from clinical_assistant.data_access import ClinicalRepository
from clinical_assistant.retrieval import ProtocolRetriever
from clinical_assistant.llm import DemoClinicalGenerator
from clinical_assistant.chains import build_clinical_chain

## Consulta estruturada

O modelo não escreve SQL. O repositório valida o identificador e executa comandos `SELECT` parametrizados, em conexão somente leitura.

In [ ]:
repository = ClinicalRepository(ROOT / "data/processed/hospital.db")
patient = repository.get_patient_context("PAC-0001")
print(patient.as_prompt_context())

## Recuperação e fontes

TF-IDF e similaridade cosseno ordenam os protocolos. Código, título, trecho e relevância acompanham cada resultado.

In [ ]:
retriever = ProtocolRetriever(ROOT / "data/raw/protocols")
sources = retriever.retrieve("dor torácica com falta de ar e eletrocardiograma", k=2)
[(item.source_id, item.score) for item in sources]

## Composição da chain

O `PromptTemplate` fixa os limites; o operador `|` encadeia o prompt e o gerador. O conteúdo recuperado é apresentado como evidência, não como instrução capaz de remover os limites.

In [ ]:
chain = build_clinical_chain(DemoClinicalGenerator())
protocol_context = "\n\n".join(f"[{item.source_id}] {item.excerpt}" for item in sources)
answer = chain.invoke({
    "question": "Quais exames estão pendentes e o que precisa ser conferido?",
    "patient_context": patient.as_prompt_context(),
    "protocol_context": protocol_context,
})
print(answer)